# Validation and Benchmarking of the Cluster-Based Cognitive Health Pipeline

This notebook is a self-contained re-evaluation of the modelling claims in this
repository. It exists because a reviewer asked for evidence that the original
notebooks do not contain, and it supersedes the numbers reported in
`d_strategyA`–`d_strategyE`, `train.ipynb` and `hybrid_inference_system_a/b`.

**What the reviewer asked for**

1. Model performance, a stated partitioning scheme, and a conventional benchmark.
2. Assurance that cluster-based oversampling did not leak across the partition.
3. Demonstration — not assertion — that the clustering reveals structure a
   conventional analysis misses.
4. Evidence for clinical utility: external validation, cluster stability, and
   proof that persona assignment improves something.

**What is different here**

| Original pipeline | This notebook |
| :--- | :--- |
| Row-level `train_test_split`, 647 persons contribute two byte-identical rows | Person-level (`UID`) grouped splits, verified by assertion |
| K-Means, scaler and one-hot fitted on all 2,889 rows | Fitted on training rows only; test rows assigned by `predict()` |
| `PredictionYear` dropped, leaving duplicate rows with different targets | `PredictionYear` retained as the 4-/9-year horizon feature |
| No cross-validation, single `random_state=42` | 5-fold grouped CV × 3 repeats with varying seeds |
| No linear benchmark anywhere | `DummyRegressor` → Ridge/Lasso/ElasticNet → RF → HistGradientBoosting |
| k=6 from a visual elbow, no validity index | Silhouette, Calinski-Harabasz, Davies-Bouldin |
| No stability analysis | Seed ARI, person-level bootstrap ARI, cross-wave ARI |
| Nearest-centroid router on unscaled income-dominated distances | Routing via the saved K-Means |
| Point estimates only | Bootstrap 95% CIs and paired bootstrap contrasts |
| No baseline ever scored on `test.csv` | Every model scored on the same 746 held-out rows |

The conclusion this notebook reaches is whatever the numbers support. It is built
so that "the pipeline adds nothing over Ridge" is a reportable outcome.

In [ ]:
import os
import sys
import io
import json
import time
import warnings
import contextlib
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Make research/utils importable exactly as the other notebooks do.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import ElasticNet, Lasso, LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    roc_auc_score,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from utils.validation import (
    SEED,
    TARGET,
    ClusterSpace,
    ClusterSpecialistRegressor,
    assert_no_group_leakage,
    bootstrap_stability,
    build_preprocessor,
    choose_k,
    cluster_share_table,
    column_kinds,
    cross_wave_stability,
    metrics_with_ci,
    oversample_by_cluster,
    paired_bootstrap_diff,
    partition_summary,
    per_cluster_metrics,
    person_level_folds,
    person_level_split,
    regression_metrics,
    route_predictions,
    seed_stability,
    split_features_target,
    stash_groups,
)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

DATA_DIR = Path("..") / "dataset" / "modified"
RESULTS_DIR = Path("..") / "results"
MODELS_DIR = Path("..") / "models" / "validated"
FIG_DIR = Path("..") / "visualization" / "f_validation"
for d in (RESULTS_DIR, MODELS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

N_SPLITS = 5
N_REPEATS = 3
N_BOOT = 2000
N_BOOT_STABILITY = 100
N_SEEDS_STABILITY = 20

results: dict[str, pd.DataFrame] = {}


def save_table(name: str, frame: pd.DataFrame) -> pd.DataFrame:
    "Persist a result table. The original pipeline wrote no results to disk."
    results[name] = frame
    frame.to_csv(RESULTS_DIR / f"{name}.csv", index=False)
    return frame


def quiet(fn, *args, **kwargs):
    "Run a chatty transformer without flooding the notebook output."
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kwargs)


print(f"numpy {np.__version__} | pandas {pd.__version__}")
import sklearn

print(f"scikit-learn {sklearn.__version__} | seed {SEED}")

## 1. Data, partitioning scheme, and the duplicate-person problem

`train.csv` and `test.csv` come from the DrivenData MHAS competition split and are
**person-disjoint**. `test.csv` is treated as an external holdout: it is loaded
here, then not touched again until section 7.

The critical structural fact, which the original notebooks did not account for:
647 training UIDs appear twice, once with a 2016 outcome and once with a 2021
outcome. Their predictor rows are byte-identical apart from `Year`,
`composite_score` and `PredictionYear`. Because every strategy notebook dropped
`Year` and `PredictionYear` and split at row level, roughly a third of each test
fold had an identical twin sitting in training — memorisation, not generalisation.

Two fixes are applied together:

* Split on `UID`, so a person is never on both sides.
* Keep `PredictionYear` (4 or 9 years to outcome) as a feature. It is known at
  prediction time, and it is what makes a person's two rows genuinely different
  observations rather than contradictory labels. The raw outcome `Year` is
  dropped, since that would leak the measurement date itself.

In [ ]:
train_raw = pd.read_csv(DATA_DIR / "train.csv")
holdout_raw = pd.read_csv(DATA_DIR / "test.csv")
print(f"train.csv  {train_raw.shape}")
print(f"test.csv   {holdout_raw.shape}")

# Quantify the duplicate-person structure the original splits ignored.
dup_cols = [c for c in train_raw.columns if c not in ("Year", "composite_score", "PredictionYear")]
dup_uids = train_raw["UID"].value_counts()
dup_uids = dup_uids[dup_uids == 2].index
twin_identical = (
    train_raw[train_raw["UID"].isin(dup_uids)]
    .groupby("UID")[dup_cols]
    .nunique()
    .max()
    .max()
)
print(
    f"\nUIDs with 2 rows: {len(dup_uids)} "
    f"({2 * len(dup_uids)} rows = {200 * len(dup_uids) / len(train_raw):.1f}% of train)"
)
print(f"max distinct values within a twin pair across {len(dup_cols)} predictors: {twin_identical}")
print("  -> 1 means every twin pair is byte-identical on all predictors")
print(f"\nYear (outcome year) distribution: {train_raw['Year'].value_counts().to_dict()}")
print(f"PredictionYear (horizon)         : {train_raw['PredictionYear'].value_counts().to_dict()}")

In [ ]:
# Drop the raw outcome year; keep PredictionYear. Stash UID before preprocessing
# removes it, otherwise the person-level split is impossible.
train_df, groups_train = stash_groups(train_raw.drop(columns=["Year"]))
holdout_df, groups_holdout = stash_groups(holdout_raw.drop(columns=["Year"]))

overlap = set(groups_train) & set(groups_holdout)
assert not overlap, f"train/holdout share {len(overlap)} UIDs"
print(f"UID overlap between train.csv and test.csv: {len(overlap)}  (person-disjoint)")

## 2. Preprocessing fitted on training rows only

Same transformers and thresholds as `c_clustering.ipynb` (drop redundant
insurance/affect columns, drop columns ≥70% missing, median/mode imputation,
drop quasi-constant numerics), with two changes made in
`utils/validation.build_preprocessor`:

* Imputation now runs **before** the dtype conversions. In the original order,
  calling `.transform()` on unseen data raised `TypeError: Cannot setitem on a
  Categorical with a new category` — which is a large part of why the original
  notebooks never separated `fit` from `transform` and instead fitted everything
  on all available rows.
* A `CategoricalAligner` forces the holdout onto the training category set and
  remaps unseen levels to the training mode, reporting how many it touched.

In [ ]:
preprocessor = build_preprocessor()
train_clean = quiet(preprocessor.fit_transform, train_df)
holdout_clean = quiet(preprocessor.transform, holdout_df)

assert list(train_clean.columns) == list(holdout_clean.columns)
assert (train_clean.dtypes.values == holdout_clean.dtypes.values).all()
assert train_clean.isnull().sum().sum() == 0
assert holdout_clean.isnull().sum().sum() == 0

X_train, y_train = split_features_target(train_clean)
X_holdout, y_holdout = split_features_target(holdout_clean)
NUMERIC, CATEGORICAL = column_kinds(X_train)

aligner = preprocessor.named_steps["6_align_categories"]
n_remapped = sum(aligner.n_unseen_.values())

print(f"cleaned train   {train_clean.shape}")
print(f"cleaned holdout {holdout_clean.shape}")
print(f"features: {X_train.shape[1]}  ({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical)")
print(f"PredictionYear retained as {X_train['PredictionYear'].dtype}: {sorted(X_train['PredictionYear'].unique())}")
print(f"missing values after imputation: train 0, holdout 0")
print(
    f"\nholdout category levels unseen in train, remapped to train mode: "
    f"{n_remapped} values across {len(aligner.n_unseen_)} columns"
)
print("  (the original all-rows fit hid this by construction)")
joblib.dump(preprocessor, MODELS_DIR / "preprocess.pkl")

In [ ]:
partitions = pd.DataFrame(
    [
        partition_summary("train (development)", y_train, groups_train),
        partition_summary("test.csv (external holdout)", y_holdout, groups_holdout),
    ]
)
save_table("01_partition_summary", partitions)
partitions

In [ ]:
# Shared design matrix: standardise numerics, one-hot categoricals. Fitted inside
# every model pipeline so it is refitted per fold and never sees holdout rows.
def make_design() -> ColumnTransformer:
    return ColumnTransformer(
        [
            ("num", StandardScaler(), NUMERIC),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"),
                CATEGORICAL,
            ),
        ]
    )


print(f"design matrix width: {make_design().fit_transform(X_train).shape[1]} columns after encoding")

## 3. Clustering: fitted on training rows, with stability statistics

Three things change relative to `c_clustering.ipynb`:

* The scaler, encoder and K-Means are bundled in a persistable `ClusterSpace`
  fitted on training rows only, with a working `predict()` for unseen data. The
  original code read `kmeans.labels_` off a fit over all rows and never saved the
  objects, which is why the inference notebooks had to approximate the router.
* `k` is chosen with silhouette, Calinski-Harabasz and Davies-Bouldin alongside
  inertia, rather than from a visual bend in a smooth elbow curve.
* Stability is measured rather than asserted: agreement across seeds, across
  person-level bootstrap resamples, and — the reviewer's specific request —
  across the 2003 and 2012 measurement waves.

In [ ]:
cluster_scaler = StandardScaler()
cluster_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")
design_train = np.hstack(
    [
        cluster_scaler.fit_transform(X_train[NUMERIC]),
        cluster_encoder.fit_transform(X_train[CATEGORICAL].astype(object)),
    ]
)
print(f"clustering design matrix: {design_train.shape}")

k_diagnostics = choose_k(design_train, k_range=range(2, 11), random_state=SEED)
save_table("02_k_selection", k_diagnostics)
k_diagnostics

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 4))
for ax, (col, title) in zip(
    axes,
    [
        ("inertia", "Inertia (elbow)"),
        ("silhouette", "Silhouette (higher better)"),
        ("calinski_harabasz", "Calinski-Harabasz (higher better)"),
        ("davies_bouldin", "Davies-Bouldin (lower better)"),
    ],
):
    ax.plot(k_diagnostics["k"], k_diagnostics[col], marker="o")
    ax.set_title(title)
    ax.set_xlabel("k")
fig.suptitle("Cluster count diagnostics, fitted on training rows only", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "k_selection.png", dpi=120, bbox_inches="tight")
plt.show()

best_sil = int(k_diagnostics.loc[k_diagnostics["silhouette"].idxmax(), "k"])
print(f"k maximising silhouette: {best_sil}")
print(f"silhouette at the originally chosen k=6: {float(k_diagnostics.loc[k_diagnostics['k'] == 6, 'silhouette'].iloc[0]):.4f}")

### 3.1 Choice of k

The original analysis used **k=6** (which then yielded one singleton cluster that
every downstream notebook discarded, leaving five). That choice is kept as the
primary specification so results stay comparable to the published strategies, and
the diagnostics above are reported honestly alongside it. If silhouette prefers a
different k, that is stated rather than hidden, and section 4's conclusions are
about whether *any* cluster-aware model beats a flat one — not about a particular k.

In [ ]:
K_PRIMARY = 6
cluster_space = ClusterSpace(
    n_clusters=K_PRIMARY, random_state=SEED, numeric_cols=NUMERIC, categorical_cols=CATEGORICAL
).fit(X_train)

assert cluster_space.n_samples_fit_ == len(X_train), "clusters must be fitted on train rows only"
labels_train = cluster_space.labels_
labels_holdout = cluster_space.predict(X_holdout)   # the predict() call absent from the original repo

joblib.dump(cluster_space, MODELS_DIR / "cluster_space.pkl")
print(f"ClusterSpace fitted on {cluster_space.n_samples_fit_} rows, {cluster_space.n_features_} dims")
print(f"train cluster sizes  : {pd.Series(labels_train).value_counts().sort_index().to_dict()}")
print(f"holdout cluster sizes: {pd.Series(labels_holdout).value_counts().sort_index().to_dict()}")

In [ ]:
shares = cluster_share_table(labels_train, labels_holdout).reset_index()
save_table("03_cluster_shares", shares)
print("Cluster prevalence, train vs holdout, using the saved K-Means predict():")
print(shares.to_string(index=False))
print(
    f"\nlargest absolute discrepancy: {shares['abs_diff'].max():.4f}"
    "\nFor contrast, hybrid_inference_system_b.ipynb routed 512/746 = 68.6% of the"
    "\nholdout to cluster 0 against 22.6% of training rows. That gap was an artifact"
    "\nof its unscaled nearest-centroid router (income terms dominate raw Euclidean"
    "\ndistance), not a real distribution shift."
)

### 3.2 Stability

Three questions, none of which the original analysis addressed:

* **Seed stability** — does refitting K-Means with a different random start give
  the same partition? Measured by pairwise adjusted Rand index (ARI) over 20 seeds.
* **Sample stability** — does the partition survive resampling the cohort?
  Measured by ARI against the reference labelling over 100 bootstrap resamples
  drawn at *person* level.
* **Cross-wave stability** — the reviewer's request. The predictors split cleanly
  into a 2003 block (`_03`) and a 2012 block (`_12`). Clustering each block
  independently and comparing by ARI asks whether the personas are a property of
  the people or of one particular measurement occasion.

ARI is 1.0 for identical partitions and ≈0 for chance agreement.

In [ ]:
t0 = time.time()
seed_ari = seed_stability(design_train, K_PRIMARY, seeds=range(N_SEEDS_STABILITY))
save_table("04_stability_seeds", seed_ari)
print(
    f"Seed stability over {N_SEEDS_STABILITY} seeds ({len(seed_ari)} pairs): "
    f"ARI mean {seed_ari['ari'].mean():.4f}, sd {seed_ari['ari'].std():.4f}, "
    f"min {seed_ari['ari'].min():.4f}, max {seed_ari['ari'].max():.4f}  [{time.time() - t0:.0f}s]"
)

In [ ]:
t0 = time.time()
boot_ari = bootstrap_stability(
    X_train, groups_train, labels_train, K_PRIMARY, n_boot=N_BOOT_STABILITY, random_state=SEED
)
save_table("05_stability_bootstrap", boot_ari)
print(
    f"Person-level bootstrap stability over {N_BOOT_STABILITY} resamples: "
    f"ARI mean {boot_ari['ari'].mean():.4f}, sd {boot_ari['ari'].std():.4f}, "
    f"2.5-97.5% [{boot_ari['ari'].quantile(0.025):.4f}, {boot_ari['ari'].quantile(0.975):.4f}]"
    f"  [{time.time() - t0:.0f}s]"
)

In [ ]:
wave = cross_wave_stability(X_train, K_PRIMARY, random_state=SEED)
wave_table = pd.DataFrame(
    [
        {
            "comparison": "2003 wave (_03) vs 2012 wave (_12)",
            "n_cols_wave_03": wave["n_cols_wave_03"],
            "n_cols_wave_12": wave["n_cols_wave_12"],
            "ari": round(wave["ari_wave03_vs_wave12"], 4),
        }
    ]
)
save_table("06_stability_cross_wave", wave_table)
print(wave_table.to_string(index=False))
print(
    "\nA low ARI here means the personas are not a stable property of the person:"
    "\nthe same cohort clustered on its 2003 measurements and on its 2012"
    "\nmeasurements produces materially different groups."
)

In [ ]:
stability_summary = pd.DataFrame(
    [
        {"check": "seed (20 seeds, pairwise)", "ari_mean": round(seed_ari["ari"].mean(), 4), "ari_sd": round(seed_ari["ari"].std(), 4)},
        {"check": "person-level bootstrap (100)", "ari_mean": round(boot_ari["ari"].mean(), 4), "ari_sd": round(boot_ari["ari"].std(), 4)},
        {"check": "cross-wave (2003 vs 2012)", "ari_mean": round(wave["ari_wave03_vs_wave12"], 4), "ari_sd": np.nan},
    ]
)
save_table("07_stability_summary", stability_summary)
stability_summary

## 4. Matched comparator ladder

Every rung sees the **same features, the same grouped folds, and the same
metrics**. This is the comparison the manuscript claims to make.

| Rung | Model | Role |
| :--- | :--- | :--- |
| 0 | `DummyRegressor(mean)` | floor; R² ≈ 0 by construction |
| 1 | Ridge / Lasso / ElasticNet | the conventional benchmark the reviewer named |
| 2 | Global `RandomForestRegressor` | flat ML, **no cluster information** |
| 3 | `HistGradientBoostingRegressor` | modern GBM reference |
| 4 | Strategy A — per-cluster specialists | original "divide and conquer" |
| 5 | Strategy B — global model + cluster feature | original meta-feature approach |
| 6 | Strategy C — cluster-balanced oversampling | original minority-boosting approach |
| 7 | Hybrid router | cluster 0 → C, small cluster 1 → B, rest → A |

**Leakage control inside the CV loop.** Cluster labels are refitted from scratch
on each training fold and applied to the validation fold by `predict()`.
Oversampling happens *after* the fold split, on training rows only — this is what
`d_strategyC.ipynb` already did correctly, and the reviewer's leakage concern on
that specific step does not hold. The genuine leak was upstream: cluster labels
derived from all rows.

**Tuning.** Linear penalties are tuned by grouped `GridSearchCV` nested inside
each training fold. Tree hyperparameters are tuned once on a held-out development
split (grouped inner CV) and then fixed, because a full nested search over 15
outer folds is not computationally worthwhile for the resolution it would add.
This is stated rather than glossed; the original pipeline did no tuning at all.

In [ ]:
dev_idx, val_idx = person_level_split(X_train, y_train, groups_train, test_size=0.2, random_state=SEED)
assert_no_group_leakage(groups_train, dev_idx, val_idx)
print(f"tuning split: {len(dev_idx)} dev rows / {len(val_idx)} val rows, no shared UIDs")

inner_cv = GroupKFold(n_splits=3)
X_dev, y_dev, g_dev = X_train.iloc[dev_idx], y_train.iloc[dev_idx], groups_train.iloc[dev_idx]

rf_grid = GridSearchCV(
    SkPipeline([("design", make_design()), ("est", RandomForestRegressor(random_state=SEED, n_jobs=-1))]),
    {
        "est__n_estimators": [300],
        "est__max_depth": [None, 12],
        "est__min_samples_leaf": [1, 5, 10],
        "est__max_features": ["sqrt", 0.3],
    },
    cv=inner_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=1,
)
t0 = time.time()
rf_grid.fit(X_dev, y_dev, groups=g_dev)
RF_PARAMS = {k.replace("est__", ""): v for k, v in rf_grid.best_params_.items()}
print(f"RF tuned in {time.time() - t0:.0f}s -> {RF_PARAMS} (dev MAE {-rf_grid.best_score_:.3f})")

hgb_grid = GridSearchCV(
    SkPipeline([("design", make_design()), ("est", HistGradientBoostingRegressor(random_state=SEED))]),
    {
        "est__max_depth": [None, 6],
        "est__learning_rate": [0.05, 0.1],
        "est__min_samples_leaf": [10, 30],
    },
    cv=inner_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=1,
)
t0 = time.time()
hgb_grid.fit(X_dev, y_dev, groups=g_dev)
HGB_PARAMS = {k.replace("est__", ""): v for k, v in hgb_grid.best_params_.items()}
print(f"HGB tuned in {time.time() - t0:.0f}s -> {HGB_PARAMS} (dev MAE {-hgb_grid.best_score_:.3f})")

save_table(
    "08_tuning",
    pd.DataFrame(
        [
            {"model": "random_forest", "params": json.dumps(RF_PARAMS), "dev_mae": round(-rf_grid.best_score_, 4)},
            {"model": "hist_gradient_boosting", "params": json.dumps(HGB_PARAMS), "dev_mae": round(-hgb_grid.best_score_, 4)},
        ]
    ),
)

In [ ]:
ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]


def linear_search(estimator, param_grid) -> GridSearchCV:
    "Grouped GridSearchCV nested inside whatever fold it is fitted on."
    return GridSearchCV(
        SkPipeline([("design", make_design()), ("est", estimator)]),
        param_grid,
        cv=GroupKFold(n_splits=3),
        scoring="neg_mean_absolute_error",
        n_jobs=1,
    )


def build_models() -> dict:
    "Fresh, unfitted comparators. Called once per fold so nothing carries over."
    return {
        "0_dummy_mean": SkPipeline([("design", make_design()), ("est", DummyRegressor(strategy="mean"))]),
        "1a_ridge": linear_search(Ridge(random_state=SEED), {"est__alpha": ALPHAS}),
        "1b_lasso": linear_search(Lasso(random_state=SEED, max_iter=5000), {"est__alpha": [0.01, 0.1, 1.0, 10.0]}),
        "1c_elasticnet": linear_search(
            ElasticNet(random_state=SEED, max_iter=5000),
            {"est__alpha": [0.01, 0.1, 1.0, 10.0], "est__l1_ratio": [0.2, 0.5, 0.8]},
        ),
        "2_random_forest_global": SkPipeline(
            [("design", make_design()), ("est", RandomForestRegressor(random_state=SEED, n_jobs=-1, **RF_PARAMS))]
        ),
        "3_hist_gradient_boosting": SkPipeline(
            [("design", make_design()), ("est", HistGradientBoostingRegressor(random_state=SEED, **HGB_PARAMS))]
        ),
    }


def fit_cluster_aware(X_tr, y_tr, lab_tr):
    "Strategies A, B and C. All cluster labels are supplied by the caller from a train-fold-only fit."
    # Strategy A: one specialist per cluster, global fallback for tiny clusters.
    strat_a = ClusterSpecialistRegressor(
        SkPipeline([("design", make_design()), ("est", RandomForestRegressor(random_state=SEED, n_jobs=-1, **RF_PARAMS))]),
        min_cluster_size=30,
    ).fit(X_tr, y_tr, lab_tr)

    # Strategy B: single global model with the cluster id as an extra feature.
    def with_cluster(frame, labels):
        out = frame.copy()
        out["cluster"] = pd.Categorical(labels, categories=sorted(set(lab_tr)))
        return out

    design_b = ColumnTransformer(
        [
            ("num", StandardScaler(), NUMERIC),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"),
                CATEGORICAL + ["cluster"],
            ),
        ]
    )
    strat_b = SkPipeline(
        [("design", design_b), ("est", RandomForestRegressor(random_state=SEED, n_jobs=-1, **RF_PARAMS))]
    ).fit(with_cluster(X_tr, lab_tr), y_tr)

    # Strategy C: cluster-balanced oversampling, applied to training rows only.
    X_bal, y_bal, _ = oversample_by_cluster(X_tr, y_tr, lab_tr, random_state=SEED)
    assert len(X_bal) >= len(X_tr), "oversampling must not shrink the training set"
    strat_c = SkPipeline(
        [("design", make_design()), ("est", RandomForestRegressor(random_state=SEED, n_jobs=-1, **RF_PARAMS))]
    ).fit(X_bal, y_bal)

    return strat_a, strat_b, strat_c, with_cluster, len(X_bal)


ROUTES = {0: "6_strategy_C_oversampled", 1: "5_strategy_B_cluster_feature"}
ROUTE_DEFAULT = "4_strategy_A_specialists"


def strategy_predictors(strat_a, strat_b, strat_c, with_cluster) -> dict:
    "Uniform (rows, labels) -> predictions interface for the router."
    return {
        "4_strategy_A_specialists": lambda rows, labels: strat_a.predict(rows, labels),
        "5_strategy_B_cluster_feature": lambda rows, labels: strat_b.predict(with_cluster(rows, labels)),
        "6_strategy_C_oversampled": lambda rows, labels: strat_c.predict(rows),
    }


print("routing map:", ROUTES, "| default:", ROUTE_DEFAULT)

In [ ]:
def evaluate_fold(X_tr, y_tr, g_tr, X_te, y_te):
    "Fit every rung on one training fold and predict the matching validation fold."
    preds: dict[str, np.ndarray] = {}

    for name, model in build_models().items():
        if isinstance(model, GridSearchCV):
            model.fit(X_tr, y_tr, groups=g_tr)
        else:
            model.fit(X_tr, y_tr)
        preds[name] = model.predict(X_te)

    # Cluster labels refitted on this training fold only, then predicted onto the
    # validation fold. This is the leakage the original notebooks contained.
    space = ClusterSpace(
        n_clusters=K_PRIMARY, random_state=SEED, numeric_cols=NUMERIC, categorical_cols=CATEGORICAL
    ).fit(X_tr)
    lab_tr, lab_te = space.labels_, space.predict(X_te)

    strat_a, strat_b, strat_c, with_cluster, n_bal = fit_cluster_aware(X_tr, y_tr, lab_tr)
    preds["4_strategy_A_specialists"] = strat_a.predict(X_te, lab_te)
    preds["5_strategy_B_cluster_feature"] = strat_b.predict(with_cluster(X_te, lab_te))
    preds["6_strategy_C_oversampled"] = strat_c.predict(X_te)

    preds["7_hybrid_router"], _ = route_predictions(
        X_te, lab_te, strategy_predictors(strat_a, strat_b, strat_c, with_cluster), ROUTES, ROUTE_DEFAULT
    )

    return preds, lab_te, n_bal


t0 = time.time()
cv_rows: list[dict] = []
fold_predictions: list[dict] = []

for repeat, fold, tr_idx, te_idx in person_level_folds(
    X_train, y_train, groups_train, n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED
):
    assert_no_group_leakage(groups_train, tr_idx, te_idx)
    X_tr, y_tr, g_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx], groups_train.iloc[tr_idx]
    X_te, y_te = X_train.iloc[te_idx], y_train.iloc[te_idx]

    preds, lab_te, n_bal = evaluate_fold(X_tr, y_tr, g_tr, X_te, y_te)
    for name, pred in preds.items():
        cv_rows.append({"repeat": repeat, "fold": fold, "model": name, **regression_metrics(y_te, pred)})
    fold_predictions.append({"repeat": repeat, "fold": fold, "y": y_te.to_numpy(), "clusters": lab_te, "preds": preds})
    print(
        f"repeat {repeat} fold {fold}: {len(tr_idx)} train / {len(te_idx)} val "
        f"(oversampled train {n_bal}) [{time.time() - t0:.0f}s]"
    )

cv_detail = pd.DataFrame(cv_rows)
save_table("09_cv_folds", cv_detail)
print(f"\ntotal CV time {time.time() - t0:.0f}s over {N_SPLITS * N_REPEATS} folds")

In [ ]:
cv_summary = (
    cv_detail.groupby("model")
    .agg(
        mae_mean=("mae", "mean"),
        mae_sd=("mae", "std"),
        rmse_mean=("rmse", "mean"),
        rmse_sd=("rmse", "std"),
        r2_mean=("r2", "mean"),
        r2_sd=("r2", "std"),
        n_folds=("mae", "size"),
    )
    .round(4)
    .reset_index()
    .sort_values("mae_mean")
)
save_table("10_cv_summary", cv_summary)
print(f"Repeated grouped CV: {N_SPLITS}-fold x {N_REPEATS} repeats, person-level, mean +/- sd across folds")
cv_summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
order = cv_summary["model"].tolist()
sns.boxplot(data=cv_detail, x="model", y="mae", order=order, ax=ax, color="#9ecae1")
sns.stripplot(data=cv_detail, x="model", y="mae", order=order, ax=ax, color="#22405a", size=4, alpha=0.7)
ax.set_title(f"Cross-validated MAE, person-level {N_SPLITS}-fold x {N_REPEATS} repeats (lower is better)")
ax.set_xlabel("")
ax.set_ylabel("MAE (composite_score points)")
ax.tick_params(axis="x", rotation=30)
for label in ax.get_xticklabels():
    label.set_ha("right")
fig.tight_layout()
fig.savefig(FIG_DIR / "cv_mae.png", dpi=120, bbox_inches="tight")
plt.show()

### 4.1 Does the cluster machinery add anything?

The question the reviewer framed as *"that has to be demonstrated rather than
assumed"*. Answered by pooling predictions across all folds and computing paired
bootstrap differences in MAE against two references: the best linear model and
the global Random Forest. Paired means both models are scored on identical
bootstrap draws of the same rows.

A negative difference favours the cluster-aware model. An interval spanning zero
means the data do not distinguish them.

In [ ]:
pooled_y = np.concatenate([f["y"] for f in fold_predictions])
pooled = {
    name: np.concatenate([f["preds"][name] for f in fold_predictions])
    for name in fold_predictions[0]["preds"]
}
pooled_clusters = np.concatenate([f["clusters"] for f in fold_predictions])

linear_names = ["1a_ridge", "1b_lasso", "1c_elasticnet"]
best_linear = min(linear_names, key=lambda n: mean_absolute_error(pooled_y, pooled[n]))
print(f"best linear benchmark: {best_linear} (pooled MAE {mean_absolute_error(pooled_y, pooled[best_linear]):.4f})")

contrast_rows = []
for reference in (best_linear, "2_random_forest_global"):
    for name in ["4_strategy_A_specialists", "5_strategy_B_cluster_feature", "6_strategy_C_oversampled", "7_hybrid_router"]:
        diff = paired_bootstrap_diff(pooled_y, pooled[name], pooled[reference], n_boot=N_BOOT, random_state=SEED)
        contrast_rows.append(
            {
                "model": name,
                "reference": reference,
                "mae_model": round(float(mean_absolute_error(pooled_y, pooled[name])), 4),
                "mae_reference": round(float(mean_absolute_error(pooled_y, pooled[reference])), 4),
                "mae_diff": round(diff["diff"], 4),
                "ci_lo": round(diff["diff_lo"], 4),
                "ci_hi": round(diff["diff_hi"], 4),
                "verdict": "model better" if diff["favours_a"] else ("reference better" if diff["favours_b"] else "inconclusive"),
            }
        )

contrasts = pd.DataFrame(contrast_rows)
save_table("11_cv_contrasts", contrasts)
print("\nPaired bootstrap MAE differences (model - reference), pooled over all CV folds:")
contrasts

In [ ]:
per_cluster = per_cluster_metrics(pooled_y, pooled["7_hybrid_router"], pooled_clusters)
per_cluster_linear = per_cluster_metrics(pooled_y, pooled[best_linear], pooled_clusters)
per_cluster_cmp = per_cluster.merge(
    per_cluster_linear[["cluster", "mae", "rmse"]], on="cluster", suffixes=("_hybrid", "_linear")
)
per_cluster_cmp["mae_advantage_hybrid"] = (
    per_cluster_cmp["mae_linear"] - per_cluster_cmp["mae_hybrid"]
).round(4)
save_table("12_cv_per_cluster", per_cluster_cmp)
print("Within-cluster error, pooled across folds. MAE and RMSE are on the outcome")
print("scale and comparable across groups; R2 is NOT -- it rescales by each group's")
print("own variance, which is why the original d_strategyA comparison of a")
print("within-cluster R2 of 0.59 against a whole-sample 0.38 was not evidence of")
print("improvement. outcome_sd is shown so that artifact is visible.")
per_cluster_cmp

## 5. Secondary binary analysis: discrimination and calibration

The reviewer asked for "accuracy, discrimination, or calibration statistics".
Those are classification quantities and are not defined for a continuous
outcome, so this section adds a pre-specified secondary framing:

* **Threshold:** the lowest quartile of the *training* `composite_score`
  distribution defines the "impaired" class. The cut is computed on training data
  and applied unchanged to the holdout — it is never refitted on the holdout.
* **Benchmark:** `LogisticRegression`, the exact comparator the reviewer named,
  against a global Random Forest classifier and a cluster-aware variant.
* **Reported:** ROC-AUC and average precision (discrimination), Brier score and a
  calibration curve (calibration), plus balanced accuracy, sensitivity and
  specificity at the 0.5 operating point.

In [ ]:
IMPAIRED_QUANTILE = 0.25
threshold = float(y_train.quantile(IMPAIRED_QUANTILE))
b_train = (y_train <= threshold).astype(int)
b_holdout = (y_holdout <= threshold).astype(int)
print(f"'impaired' = composite_score <= {threshold:.1f} (training {IMPAIRED_QUANTILE:.0%} quantile)")
print(f"prevalence: train {b_train.mean():.4f}, holdout {b_holdout.mean():.4f}")


def classification_metrics(y_true, proba, label: str) -> dict:
    pred = (proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "model": label,
        "n": int(len(y_true)),
        "roc_auc": round(float(roc_auc_score(y_true, proba)), 4),
        "avg_precision": round(float(average_precision_score(y_true, proba)), 4),
        "brier": round(float(brier_score_loss(y_true, proba)), 4),
        "accuracy": round(float(accuracy_score(y_true, pred)), 4),
        "balanced_accuracy": round(float(balanced_accuracy_score(y_true, pred)), 4),
        "sensitivity": round(float(tp / (tp + fn)) if (tp + fn) else np.nan, 4),
        "specificity": round(float(tn / (tn + fp)) if (tn + fp) else np.nan, 4),
    }


def build_classifiers() -> dict:
    return {
        "0_dummy_prior": SkPipeline([("design", make_design()), ("est", DummyClassifier(strategy="prior"))]),
        "1_logistic_regression": GridSearchCV(
            SkPipeline(
                [("design", make_design()), ("est", LogisticRegression(max_iter=5000, random_state=SEED))]
            ),
            {"est__C": [0.01, 0.1, 1.0, 10.0]},
            cv=GroupKFold(n_splits=3),
            scoring="roc_auc",
            n_jobs=1,
        ),
        "2_random_forest_global": SkPipeline(
            [
                ("design", make_design()),
                (
                    "est",
                    RandomForestClassifier(
                        n_estimators=300, random_state=SEED, n_jobs=-1, class_weight="balanced"
                    ),
                ),
            ]
        ),
    }

In [ ]:
t0 = time.time()
clf_rows, clf_pooled = [], {}
pooled_binary_y = []

for repeat, fold, tr_idx, te_idx in person_level_folds(
    X_train, b_train, groups_train, n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED
):
    assert_no_group_leakage(groups_train, tr_idx, te_idx)
    X_tr, b_tr, g_tr = X_train.iloc[tr_idx], b_train.iloc[tr_idx], groups_train.iloc[tr_idx]
    X_te, b_te = X_train.iloc[te_idx], b_train.iloc[te_idx]
    pooled_binary_y.append(b_te.to_numpy())

    for name, model in build_classifiers().items():
        if isinstance(model, GridSearchCV):
            model.fit(X_tr, b_tr, groups=g_tr)
        else:
            model.fit(X_tr, b_tr)
        proba = model.predict_proba(X_te)[:, 1]
        clf_rows.append({"repeat": repeat, "fold": fold, **classification_metrics(b_te, proba, name)})
        clf_pooled.setdefault(name, []).append(proba)

    # Cluster-aware variant: cluster id added as a feature, labels from this fold only.
    space = ClusterSpace(
        n_clusters=K_PRIMARY, random_state=SEED, numeric_cols=NUMERIC, categorical_cols=CATEGORICAL
    ).fit(X_tr)
    lab_tr, lab_te = space.labels_, space.predict(X_te)

    def augment(frame, labels):
        out = frame.copy()
        out["cluster"] = pd.Categorical(labels, categories=sorted(set(lab_tr)))
        return out

    design_c = ColumnTransformer(
        [
            ("num", StandardScaler(), NUMERIC),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"), CATEGORICAL + ["cluster"]),
        ]
    )
    clf_cluster = SkPipeline(
        [("design", design_c), ("est", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, class_weight="balanced"))]
    ).fit(augment(X_tr, lab_tr), b_tr)
    proba = clf_cluster.predict_proba(augment(X_te, lab_te))[:, 1]
    clf_rows.append({"repeat": repeat, "fold": fold, **classification_metrics(b_te, proba, "3_random_forest_cluster_aware")})
    clf_pooled.setdefault("3_random_forest_cluster_aware", []).append(proba)

clf_detail = pd.DataFrame(clf_rows)
save_table("13_binary_cv_folds", clf_detail)
print(f"binary CV done in {time.time() - t0:.0f}s")

clf_summary = (
    clf_detail.groupby("model")[
        ["roc_auc", "avg_precision", "brier", "accuracy", "balanced_accuracy", "sensitivity", "specificity"]
    ]
    .agg(["mean", "std"])
    .round(4)
)
clf_summary.columns = ["_".join(c) for c in clf_summary.columns]
clf_summary = clf_summary.reset_index().sort_values("roc_auc_mean", ascending=False)
save_table("14_binary_cv_summary", clf_summary)
clf_summary

In [ ]:
pooled_b = np.concatenate(pooled_binary_y)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, chunks in clf_pooled.items():
    proba = np.concatenate(chunks)
    if name.startswith("0_"):
        continue
    frac_pos, mean_pred = calibration_curve(pooled_b, proba, n_bins=10, strategy="quantile")
    axes[0].plot(mean_pred, frac_pos, marker="o", label=f"{name} (Brier {brier_score_loss(pooled_b, proba):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
axes[0].set_xlabel("mean predicted probability")
axes[0].set_ylabel("observed fraction impaired")
axes[0].set_title("Calibration, pooled across CV folds")
axes[0].legend(fontsize=8)

from sklearn.metrics import roc_curve

for name, chunks in clf_pooled.items():
    if name.startswith("0_"):
        continue
    proba = np.concatenate(chunks)
    fpr, tpr, _ = roc_curve(pooled_b, proba)
    axes[1].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(pooled_b, proba):.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("false positive rate")
axes[1].set_ylabel("true positive rate")
axes[1].set_title("Discrimination (ROC), pooled across CV folds")
axes[1].legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "calibration_roc.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. Triage classifier, reframed

`d_strategyD.ipynb` reported 93% accuracy for predicting cluster membership and
concluded that this "proves the groups are clinically distinct". It does not. The
classifier was given the same ~157 features that K-Means used to draw the
partition, so it was recovering a deterministic geometric boundary from its own
inputs. High accuracy there is a statement about the learnability of a Voronoi
cell, not about clinical validity.

Two things make the number interpretable:

* a majority-class baseline, so 93% has a reference point;
* a version restricted to features a person at intake could actually supply —
  age, sex, education, marital status, self-rated health — which is what the
  README claims the triage step uses.

In [ ]:
INTAKE_PATTERNS = ("Age_", "Gender", "Education", "Married", "Marriages", "GlobalHealth", "Urban")
intake_cols = [c for c in X_train.columns if any(c.startswith(p) for p in INTAKE_PATTERNS)]
print(f"intake-observable features ({len(intake_cols)}): {intake_cols}")

triage_rows = []
for label, cols in [("all_features (as published)", list(X_train.columns)), ("intake_observable_only", intake_cols)]:
    fold_acc, fold_bal = [], []
    for repeat, fold, tr_idx, te_idx in person_level_folds(
        X_train, y_train, groups_train, n_splits=N_SPLITS, n_repeats=1, random_state=SEED
    ):
        X_tr, X_te = X_train.iloc[tr_idx], X_train.iloc[te_idx]
        space = ClusterSpace(
            n_clusters=K_PRIMARY, random_state=SEED, numeric_cols=NUMERIC, categorical_cols=CATEGORICAL
        ).fit(X_tr)
        lab_tr, lab_te = space.labels_, space.predict(X_te)

        num_sub = [c for c in NUMERIC if c in cols]
        cat_sub = [c for c in CATEGORICAL if c in cols]
        design = ColumnTransformer(
            [
                ("num", StandardScaler(), num_sub),
                ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"), cat_sub),
            ]
        )
        clf = SkPipeline(
            [("design", design), ("est", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, class_weight="balanced"))]
        ).fit(X_tr[cols], lab_tr)
        pred = clf.predict(X_te[cols])
        fold_acc.append(accuracy_score(lab_te, pred))
        fold_bal.append(balanced_accuracy_score(lab_te, pred))

    triage_rows.append(
        {
            "feature_set": label,
            "n_features": len(cols),
            "accuracy_mean": round(float(np.mean(fold_acc)), 4),
            "accuracy_sd": round(float(np.std(fold_acc, ddof=1)), 4),
            "balanced_accuracy_mean": round(float(np.mean(fold_bal)), 4),
        }
    )

majority = float(pd.Series(labels_train).value_counts(normalize=True).max())
triage_rows.append(
    {
        "feature_set": "majority-class baseline",
        "n_features": 0,
        "accuracy_mean": round(majority, 4),
        "accuracy_sd": np.nan,
        "balanced_accuracy_mean": round(1.0 / K_PRIMARY, 4),
    }
)
triage = pd.DataFrame(triage_rows)
save_table("15_triage_reframed", triage)
print("\nRecovering K-Means labels. Note this measures boundary learnability, not")
print("phenotype validity: the 'all_features' row predicts the partition from the")
print("very inputs that defined it.")
triage

## 7. External holdout: every model, the same 746 rows, evaluated once

Each rung is refitted on all of `train.csv` and scored on `test.csv`, which is
person-disjoint from training and has not been used anywhere above. This is the
comparison the manuscript asserts and the original code never performed: no
baseline of any kind was ever run against `test.csv`.

Bootstrap 95% CIs accompany every number, and the paired contrasts repeat the
section 4.1 test on the holdout.

In [ ]:
t0 = time.time()
holdout_preds: dict[str, np.ndarray] = {}

for name, model in build_models().items():
    if isinstance(model, GridSearchCV):
        model.fit(X_train, y_train, groups=groups_train)
        joblib.dump(model.best_estimator_, MODELS_DIR / f"{name}.pkl")
    else:
        model.fit(X_train, y_train)
        joblib.dump(model, MODELS_DIR / f"{name}.pkl")
    holdout_preds[name] = model.predict(X_holdout)
    print(f"fitted {name} [{time.time() - t0:.0f}s]")

strat_a, strat_b, strat_c, with_cluster, n_bal = fit_cluster_aware(X_train, y_train, labels_train)
holdout_preds["4_strategy_A_specialists"] = strat_a.predict(X_holdout, labels_holdout)
holdout_preds["5_strategy_B_cluster_feature"] = strat_b.predict(with_cluster(X_holdout, labels_holdout))
holdout_preds["6_strategy_C_oversampled"] = strat_c.predict(X_holdout)

holdout_preds["7_hybrid_router"], route_used = route_predictions(
    X_holdout,
    labels_holdout,
    strategy_predictors(strat_a, strat_b, strat_c, with_cluster),
    ROUTES,
    ROUTE_DEFAULT,
)

for obj, fname in [(strat_a, "strategy_A.pkl"), (strat_b, "strategy_B.pkl"), (strat_c, "strategy_C.pkl")]:
    joblib.dump(obj, MODELS_DIR / fname)

print(f"\noversampled training rows: {n_bal} (from {len(X_train)})")
print(f"holdout routing: {pd.Series(route_used).value_counts().to_dict()}")
print(f"all models fitted in {time.time() - t0:.0f}s")

In [ ]:
assert all(len(p) == len(y_holdout) for p in holdout_preds.values()), "every model must score all holdout rows"

holdout_rows = []
for name, pred in holdout_preds.items():
    holdout_rows.append({"model": name, **metrics_with_ci(y_holdout, pred, n_boot=N_BOOT, random_state=SEED)})
holdout_table = pd.DataFrame(holdout_rows).sort_values("mae").reset_index(drop=True)
save_table("16_holdout_results", holdout_table)
print(f"External holdout: {len(y_holdout)} person-disjoint rows, 95% bootstrap CIs")
holdout_table

In [ ]:
best_linear_holdout = min(linear_names, key=lambda n: mean_absolute_error(y_holdout, holdout_preds[n]))
holdout_contrasts = []
for reference in (best_linear_holdout, "2_random_forest_global"):
    for name in ["4_strategy_A_specialists", "5_strategy_B_cluster_feature", "6_strategy_C_oversampled", "7_hybrid_router"]:
        diff = paired_bootstrap_diff(
            y_holdout, holdout_preds[name], holdout_preds[reference], n_boot=N_BOOT, random_state=SEED
        )
        holdout_contrasts.append(
            {
                "model": name,
                "reference": reference,
                "mae_model": round(float(mean_absolute_error(y_holdout, holdout_preds[name])), 4),
                "mae_reference": round(float(mean_absolute_error(y_holdout, holdout_preds[reference])), 4),
                "mae_diff": round(diff["diff"], 4),
                "ci_lo": round(diff["diff_lo"], 4),
                "ci_hi": round(diff["diff_hi"], 4),
                "verdict": "model better" if diff["favours_a"] else ("reference better" if diff["favours_b"] else "inconclusive"),
            }
        )
holdout_contrast_table = pd.DataFrame(holdout_contrasts)
save_table("17_holdout_contrasts", holdout_contrast_table)
print("Paired bootstrap MAE differences on the external holdout (model - reference):")
holdout_contrast_table

In [ ]:
holdout_per_cluster = per_cluster_metrics(y_holdout, holdout_preds["7_hybrid_router"], labels_holdout)
holdout_per_cluster_lin = per_cluster_metrics(y_holdout, holdout_preds[best_linear_holdout], labels_holdout)
cmp_cluster = holdout_per_cluster.merge(
    holdout_per_cluster_lin[["cluster", "mae"]], on="cluster", suffixes=("_hybrid", "_linear")
)
cmp_cluster["mae_advantage_hybrid"] = (cmp_cluster["mae_linear"] - cmp_cluster["mae_hybrid"]).round(4)
save_table("18_holdout_per_cluster", cmp_cluster)
cmp_cluster

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = holdout_table.sort_values("mae")
ax.errorbar(
    plot_df["mae"],
    range(len(plot_df)),
    xerr=[plot_df["mae"] - plot_df["mae_lo"], plot_df["mae_hi"] - plot_df["mae"]],
    fmt="o",
    color="#22405a",
    capsize=4,
)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df["model"])
ax.invert_yaxis()
ax.set_xlabel("MAE on the 746-row external holdout (95% bootstrap CI)")
ax.set_title("Matched comparison on held-out, person-disjoint data")
ax.axvline(plot_df["mae"].min(), color="#b3492d", ls="--", lw=1, label="best model")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "holdout_mae.png", dpi=120, bbox_inches="tight")
plt.show()

### 7.1 Relation to the previously published numbers

The manuscript's headline was "hybrid MAE 34.76 / R² 0.481 versus baseline
MAE 35.19 / R² 0.380". Those two numbers are not comparable:

| | published hybrid | published "baseline" |
| :--- | :--- | :--- |
| source | `hybrid_inference_system_b.ipynb` | `train.ipynb` (AutoGluon) |
| rows scored | 746 (`test.csv`) | 578 (a random split of `train.csv`) |
| features | 157 raw | 67 engineered |
| `Year` / `PredictionYear` | dropped | retained |
| routing | unscaled nearest-centroid | n/a |

They differ in the evaluation set, the feature space and the information set. The
table in section 7 replaces both with a single matched comparison.

In [ ]:
historical = pd.DataFrame(
    [
        {"source": "hybrid_inference_system_b.ipynb", "reported_mae": 34.76, "reported_r2": 0.481,
         "n_rows": 746, "evaluation_set": "test.csv", "features": "157 raw", "comparable": False},
        {"source": "train.ipynb AutoGluon WeightedEnsemble_L2", "reported_mae": 35.19, "reported_r2": 0.3797,
         "n_rows": 578, "evaluation_set": "random split of train.csv", "features": "67 engineered", "comparable": False},
        {"source": "d_strategyA.ipynb (best cluster)", "reported_mae": 29.98, "reported_r2": 0.593,
         "n_rows": 167, "evaluation_set": "within-cluster split of train.csv", "features": "157 raw", "comparable": False},
    ]
)
save_table("19_historical_numbers", historical)
print("Previously published figures, recorded for the rebuttal as non-comparable:")
historical

## 8. Summary

The cells below assemble the findings into one table. Everything is read back
from the persisted result files, so nothing here is hand-typed — which was a
problem with the original README, where the per-cluster "key results" were the
best value picked across three notebooks that each used a different random split.

In [ ]:
best_overall = holdout_table.iloc[0]
best_cluster_aware = (
    holdout_table[holdout_table["model"].str.startswith(("4_", "5_", "6_", "7_"))].sort_values("mae").iloc[0]
)
best_conventional = (
    holdout_table[holdout_table["model"].str.startswith(("1", "2_", "3_"))].sort_values("mae").iloc[0]
)

findings = pd.DataFrame(
    [
        {"question": "Best model on the external holdout",
         "answer": f"{best_overall['model']}: MAE {best_overall['mae']} [{best_overall['mae_lo']}, {best_overall['mae_hi']}], R2 {best_overall['r2']}"},
        {"question": "Best conventional benchmark",
         "answer": f"{best_conventional['model']}: MAE {best_conventional['mae']}, R2 {best_conventional['r2']}"},
        {"question": "Best cluster-aware model",
         "answer": f"{best_cluster_aware['model']}: MAE {best_cluster_aware['mae']}, R2 {best_cluster_aware['r2']}"},
        {"question": "Does any cluster-aware model beat the conventional benchmark on the holdout?",
         "answer": "; ".join(
             f"{r['model']} vs {r['reference']}: {r['verdict']} (diff {r['mae_diff']} [{r['ci_lo']}, {r['ci_hi']}])"
             for _, r in holdout_contrast_table.iterrows()
         )},
        {"question": "Cluster stability across seeds",
         "answer": f"ARI {seed_ari['ari'].mean():.4f} +/- {seed_ari['ari'].std():.4f}"},
        {"question": "Cluster stability under person-level bootstrap",
         "answer": f"ARI {boot_ari['ari'].mean():.4f} +/- {boot_ari['ari'].std():.4f}"},
        {"question": "Cluster stability across measurement waves (2003 vs 2012)",
         "answer": f"ARI {wave['ari_wave03_vs_wave12']:.4f}"},
        {"question": "Does the saved K-Means reproduce training cluster shares on the holdout?",
         "answer": f"yes, max absolute discrepancy {shares['abs_diff'].max():.4f} (the old router's 69%-to-cluster-0 collapse was a router artifact)"},
        {"question": "Discrimination for the pre-specified binary outcome",
         "answer": "; ".join(
             f"{r['model']} AUC {r['roc_auc_mean']:.4f}, Brier {r['brier_mean']:.4f}"
             for _, r in clf_summary.iterrows()
         )},
        {"question": "Triage classifier, reframed",
         "answer": "; ".join(f"{r['feature_set']}: accuracy {r['accuracy_mean']}" for _, r in triage.iterrows())},
        {"question": "Was the cluster oversampling leaking?",
         "answer": "No. d_strategyC.ipynb split before resampling and touched training rows only. The real leak was upstream: K-Means, the scaler and the one-hot encoding were fitted on all rows, and splits were row-level so duplicate-UID twins straddled the boundary."},
    ]
)
save_table("20_findings", findings)
pd.set_option("display.max_colwidth", 200)
findings

In [ ]:
print("Persisted result tables:")
for path in sorted(RESULTS_DIR.glob("*.csv")):
    print(f"  {path.name:38s} {len(pd.read_csv(path)):4d} rows")
print("\nPersisted models:")
for path in sorted(MODELS_DIR.glob("*.pkl")):
    print(f"  {path.name:38s} {path.stat().st_size / 1e6:7.2f} MB")
print("\nFigures:")
for path in sorted(FIG_DIR.glob("*.png")):
    print(f"  {path.name}")

## 9. What this supports in a rebuttal

Read `results/20_findings.csv` for the numbers. Regardless of which way they come
out, the following are now defensible statements about the method rather than
assertions:

1. **Partitioning is stated and person-level.** Section 1 documents the scheme;
   assertions in sections 4–7 fail loudly if a `UID` ever spans a fold.
2. **A conventional benchmark exists.** Ridge, Lasso, ElasticNet and logistic
   regression are all present, tuned, and scored on the same rows as everything else.
3. **Performance is reported with uncertainty.** Repeated grouped CV with
   fold-level spread, bootstrap CIs on every holdout number, and paired bootstrap
   contrasts for the comparisons that carry the argument.
4. **The oversampling leakage concern is answered directly:** it did not occur.
   Two leaks the reviewer did not identify are documented and fixed.
5. **Cluster stability is measured** across seeds, person-level bootstrap
   resamples, and the 2003/2012 waves.
6. **Claims that do not survive should be dropped.** In particular: the triage
   classifier's 93% is not evidence of phenotype validity; within-cluster R²
   cannot be compared to whole-sample R²; and if section 7 shows the hybrid does
   not beat Ridge with a CI excluding zero, the methodological contribution as
   currently framed is not supported and the paper needs a different claim.